# Phase 14A — SQL Generator Fine-tuning (Qwen2.5-Coder-7B-Instruct)

Fine-tunes `Qwen2.5-Coder-7B-Instruct` via LoRA to generate SQL queries
given the full pipeline context: enriched schema + key fields + SAR examples.

## What this notebook does

1. **Build training data** — for each of the 6748 CoT entries:
   - Retrieves top-3 structurally similar Q-SQL pairs from ChromaDB (SAR)
   - Formats a prompt: schema → key_fields → 3 SAR examples → question
   - Label: ground-truth SQL
2. **Fine-tune** — LoRA SFT on Qwen2.5-Coder-7B-Instruct (3 epochs)
3. **Save** checkpoint to Google Drive

## Prompt format (what the model learns)

```
System: You are an expert SQL query writer...

User:
## Database Schema
# Table: singer
[(singer_id:INT, ...), (name:TEXT, Examples: [Ed Sheeran, Adele]), ...]

## Key Fields
singer.country, singer.age

## Similar Examples
Example 1:
Q: How many singers are older than 25?
SQL: SELECT COUNT(*) FROM singer WHERE age > 25

## Question
How many singers are from France?

Assistant: SELECT COUNT(*) FROM singer WHERE country = 'France'
```

## Prerequisites
- Phase 12A ✅: `sar_sql/sar_model.pt` on Drive
- Phase 13 ✅: `indexes/chroma_sql/` on Drive (only needed if building data on Colab)
- `sql_generator_train.jsonl` uploaded to `MyDrive/codegen/generator_data/`
  (built locally in Phase 14A — Cell 4 skips the rebuild if present)

> **Requires A100** for efficient training (~45–60 min full run, checkpoint 1 in
> ~15–20 min). T4 works via 4-bit QLoRA but is slower (~1.5–2.5 hrs).
> Runtime → Change runtime type → A100.

## Cell 1 — Mount Drive and set paths

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE     = '/content/drive/MyDrive/codegen'
SAR_MODEL      = f'{DRIVE_BASE}/checkpoints/sar_sql/sar_model.pt'
CHROMA_SQL_DIR = f'{DRIVE_BASE}/indexes/chroma_sql'
GEN_OUT_DIR    = f'{DRIVE_BASE}/checkpoints/generator_sql'
GEN_DATA_DIR   = f'{DRIVE_BASE}/generator_data'
GEN_DATA_FILE  = f'{GEN_DATA_DIR}/sql_generator_train.jsonl'

os.makedirs(GEN_OUT_DIR,  exist_ok=True)
os.makedirs(GEN_DATA_DIR, exist_ok=True)

# Verify prerequisites
for path, label in [
    (SAR_MODEL,                    'SAR SQL model'),
    (CHROMA_SQL_DIR,               'ChromaDB SQL index'),
]:
    exists = os.path.exists(path)
    size   = f'{os.path.getsize(path)/1e6:.1f} MB' if exists and os.path.isfile(path) else ''
    print(f'{label}: {"✅" if exists else "❌"} {size}')

import torch
print(f'\nGPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')
USE_A100 = torch.cuda.is_available() and 'A100' in torch.cuda.get_device_name(0)
print(f'A100 mode: {USE_A100}')

## Cell 2 — Clone / update repo

In [ ]:
%%bash
set -e
REPO="/content/Codegen"
BRANCH="phase/14a-generator-sql"

if [ -d "$REPO/.git" ]; then
    cd "$REPO" && git fetch origin && git checkout $BRANCH && git pull origin $BRANCH
else
    git clone https://github.com/kethansplunk/Codegen.git "$REPO"
    cd "$REPO" && git checkout $BRANCH
fi
echo "Branch : $(git branch --show-current)"
echo "Commit : $(git log --oneline -1)"

## Cell 3 — Install dependencies

In [ ]:
# The latest TRL removed DataCollatorForCompletionOnlyLM and changed the
# SFTTrainer signature (dataset_text_field/max_seq_length → SFTConfig,
# tokenizer → processing_class). Pin a coherent late-2024 stack whose API
# matches src/generator/train.py. bitsandbytes stays latest for CUDA compat.
!pip install -q chromadb "FlagEmbedding==1.2.9" \
    "transformers==4.45.2" "trl==0.11.4" "peft==0.13.2" \
    "accelerate==1.0.1" "datasets==3.0.1" bitsandbytes

# Verify the pinned API surface is importable
import importlib, trl
importlib.reload(trl)
from trl import DataCollatorForCompletionOnlyLM, SFTTrainer   # must not raise
print(f'trl {trl.__version__} ✅  DataCollatorForCompletionOnlyLM present')
print('Dependencies installed ✅')
print('\nℹ️  Training (Cell 6) runs in a fresh subprocess, so it picks up these '
      'pinned versions automatically — no runtime restart needed.\n'
      'If you later run Cell 8 (in-kernel smoke test) and hit a version error, '
      'Runtime → Restart session, then re-run Cells 1, 2, 8.')

## Cell 4 — Build SQL generator training data

For each of the 6748 CoT training entries:
1. Query ChromaDB for top-3 structurally similar SQL examples (SAR)
2. Format: schema + key_fields + 3 SAR examples + question → SQL
3. Save to `sql_generator_train.jsonl` on Drive

**Skips if already built** (resumes from checkpoint if interrupted).

Expected time: ~15-20 minutes (BGE encodes each question once)

In [ ]:
import os, sys
sys.path.insert(0, '/content/Codegen')

# Check if already built
if os.path.exists(GEN_DATA_FILE):
    with open(GEN_DATA_FILE) as f:
        n = sum(1 for _ in f)
    print(f'Training data already exists: {n} entries — skipping build ✅')
else:
    import subprocess, shlex
    cmd = (
        f'python -m scripts.build_generator_training_data '
        f'--cot        /content/Codegen/Data/cot_data/sql_cot_train.json '
        f'--chroma_dir {CHROMA_SQL_DIR} '
        f'--sar_model  {SAR_MODEL} '
        f'--out        {GEN_DATA_FILE}'
    )
    result = subprocess.run(shlex.split(cmd), cwd='/content/Codegen',
                            capture_output=False, text=True)
    if result.returncode != 0:
        raise RuntimeError('Training data build failed')

## Cell 5 — Inspect training data

Verify the format before training.

In [ ]:
import json

with open(GEN_DATA_FILE) as f:
    samples = [json.loads(line) for line in f]

print(f'Training examples: {len(samples)}')
print(f'\n=== Sample entry (truncated) ===')
print(samples[0]['text'][:1200])
print('...')

## Cell 6 — Fine-tune SQL Generator

Config is auto-selected from `USE_A100` (set in Cell 1).

| | A100 | T4 |
|---|---|---|
| Precision | bf16 LoRA | 4-bit QLoRA |
| Batch × accum | 4 × 4 = **16** | 1 × 16 = **16** |
| Max seq len | 2048 | 1024 |
| Steps / epoch | ~422 | ~422 |
| Total steps (3 epochs) | ~1266 | ~1266 |
| ~Time / epoch | ~15–20 min | ~35–45 min |
| **Checkpoint 1 (end of epoch 1)** | **~15–20 min** | **~35–45 min** |
| Full run | ~45–60 min | ~1.5–2.5 hrs |

> Step count = ceil(6748 / 16) = **422 steps/epoch**. The per-step time above is an
> estimate — the callback prints the **measured** ETA after the first few steps.

Only the SQL tokens (after `<|im_start|>assistant\n`) receive loss — the prompt is masked.

### Live progress (printed by `ProgressCallback` in `src/generator/train.py`)
- `[TRAIN] plan | ...` — up front: total steps, steps/epoch, when checkpoint 1 saves
- `[TRAIN] 10% | step 130/1266 | elapsed 3.2m | ETA 28.5m | 1.48s/step` — every 10%, **measured** ETA
- `[CKPT] saved | step 422 | epoch 1.00 | elapsed 14.2m` — each checkpoint write

### Checkpoints
`save_strategy=epoch` → **one checkpoint per epoch = 3 checkpoints**, written to
`checkpoints/generator_sql/checkpoint-*/` on Drive. Checkpoint 1 lands at the end of
epoch 1 (~step 422). If Colab disconnects mid-run, the last completed epoch's
checkpoint survives on Drive.

In [ ]:
import os, subprocess

# Make the Drive training file visible at the repo's expected relative path too
os.makedirs('/content/Codegen/Data/generator_data', exist_ok=True)
link = '/content/Codegen/Data/generator_data/sql_generator_train.jsonl'
if not os.path.exists(link):
    os.symlink(GEN_DATA_FILE, link)

# Build the command from the Python vars (a %%bash cell cannot see them)
cmd = [
    'python', '-u', '-m', 'src.generator.train',   # -u = unbuffered, so lines stream
    '--data', GEN_DATA_FILE,
    '--out',  GEN_OUT_DIR,
]
if USE_A100:
    cmd.append('--use_a100')

print('Running:', ' '.join(cmd), '\n', flush=True)

# Stream stdout+stderr line-by-line via print() so it reliably shows in the
# Colab cell (plain subprocess.run inherits fds that Colab does NOT capture).
proc = subprocess.Popen(
    cmd, cwd='/content/Codegen', text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
)
tail = []
for line in proc.stdout:
    print(line, end='', flush=True)
    tail.append(line)
    tail = tail[-40:]            # keep only the last 40 lines for the error msg
proc.wait()

if proc.returncode != 0:
    raise RuntimeError(
        f'Training failed (exit {proc.returncode}). Last {len(tail)} lines:\n'
        + ''.join(tail)
    )
print('\nTraining complete ✅')

## Cell 7 — Verify saved checkpoint

In [ ]:
import os

print(f'Checkpoint directory: {GEN_OUT_DIR}')
files = os.listdir(GEN_OUT_DIR) if os.path.exists(GEN_OUT_DIR) else []
total_mb = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, fs in os.walk(GEN_OUT_DIR)
    for f in fs
) / 1e6 if files else 0

print(f'Files  : {len(files)}')
print(f'Size   : {total_mb:.1f} MB')
for fname in sorted(files):
    print(f'  {fname}')

## Cell 8 — Smoke test: generate SQL for a sample question

In [ ]:
import sys
sys.path.insert(0, '/content/Codegen')

from src.generator.infer import GeneratorInfer
from src.sar.infer import ChromaSARRetriever

# Load retriever
retriever = ChromaSARRetriever(
    model_path=SAR_MODEL,
    chroma_dir=CHROMA_SQL_DIR,
    collection_name='sar_sql',
)

# Load generator
gen = GeneratorInfer(GEN_OUT_DIR, n_candidates=1, temperature=0.0)

# Test
question   = 'How many singers are there in each country?'
schema     = '# Table: singer\n[(singer_id:INT), (name:TEXT), (country:TEXT), (age:INT)]'
key_fields = ['singer.country']
examples   = retriever.retrieve(question, top_k=3)

sql = gen.generate(question, schema, key_fields, examples)[0]
print(f'Question : {question}')
print(f'SQL      : {sql}')